# Demo 2 — Apply targeted transformations

**Learning objectives**

- Apply documented sentinel, string, category, numeric, date, and exact-duplicate rules to a working copy.
- Preserve uncertainty with missing values and a review flag instead of inventing replacements.
- Explain why adjacent-row filling is invalid without entity and order semantics.

Colab is the default launch experience; local Jupyter runs the same cells. See `DEMO_GUIDE.md` for launch and rehearsal instructions. GitHub source opened in Colab is not automatically updated by edits in the Colab tab.

Compatibility candidate: Python 3.12.13, NumPy 2.0.2, pandas 3.0.3. This is not the final course lock until fresh local and Colab certification is complete. The fixture contains invented teaching records only.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

PANDAS_CANDIDATE = "3.0.3"
try:
    installed_pandas = version("pandas")
except PackageNotFoundError:
    installed_pandas = None
if installed_pandas != PANDAS_CANDIDATE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", f"pandas=={PANDAS_CANDIDATE}"],
        check=True,
    )

import numpy as np
import pandas as pd

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

## Resolve the pinned source and preserve raw state

This is the same six-row source audited in Demo 1. The exact-duplicate keep mask is derived from untouched raw rows before normalization or conversion.

In [ ]:
from hashlib import sha256
from pathlib import Path

SOURCE_RELATIVE_PATH = Path("05") / "demo" / "data" / "supplied_people_raw.csv"
EXPECTED_SHA256 = "7b3223154756aa59f2f00027ddbadaa225eeee51ad75d0df91de1fd8d14abe2d"
SUPPLIED_SOURCE_BYTES = (
    b"record_id,full_name,site,status,age_text,visit_date\n"
    b"R001, Alice Smith , north,Active,34,2026-01-15\n"
    b"R002,BOB JONES,North,active,unknown,2026-02-30\n"
    b"R002,BOB JONES,North,active,unknown,2026-02-30\n"
    b"R003, Carla Ruiz ,SOUTH,pending,-9,2026-03-01\n"
    b"R004,,south,NA,45,\n"
    b"R005,Evan Li,west,complete,52,2026-02-14\n"
)


def find_course_file(start, relative_path):
    current = start.resolve()
    while True:
        candidate = current / relative_path
        if candidate.is_file():
            return candidate
        if current.parent == current:
            return None
        current = current.parent


DATA_PATH = find_course_file(Path.cwd(), SOURCE_RELATIVE_PATH)
if DATA_PATH is None:
    data_dir = Path.cwd() / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    DATA_PATH = data_dir / "supplied_people_raw.csv"
    DATA_PATH.write_bytes(SUPPLIED_SOURCE_BYTES)

assert sha256(DATA_PATH.read_bytes()).hexdigest() == EXPECTED_SHA256
raw = pd.read_csv(DATA_PATH, keep_default_na=False)
raw_snapshot = raw.copy(deep=True)
exact_duplicate_keep_mask = ~raw.duplicated(keep="first")
working = raw.copy(deep=True)
print("Input:", DATA_PATH)
raw

## Reject an invalid adjacent-row fill

Forward fill copies the preceding row's value. Here the empty visit date belongs to `R004`, while the preceding row belongs to `R003`; rows are different people and have no within-person sequence. The preview makes that failure visible but is never assigned to `working`.

In [ ]:
invalid_fill_preview = raw[["record_id", "visit_date"]].copy()
invalid_fill_preview["would_be_filled_date"] = (
    invalid_fill_preview["visit_date"].replace({"": pd.NA}).ffill()
)
r004_preview = invalid_fill_preview.loc[raw["record_id"].eq("R004")].iloc[0]
assert r004_preview["would_be_filled_date"] == "2026-03-01"
invalid_fill_preview

## Convert only documented sentinels

Each replacement follows the source dictionary from Demo 1. It is not a universal list of missing tokens.

In [ ]:
working["full_name"] = working["full_name"].replace({"": pd.NA})
working["status"] = working["status"].replace({"NA": pd.NA})
working["age_text"] = working["age_text"].replace({"unknown": pd.NA, "-9": pd.NA})
working["visit_date"] = working["visit_date"].replace({"": pd.NA})
working

## Normalize bounded string fields

Normalization maps equivalent representations to one documented form. These operations are restricted to columns whose meanings and allowed values are already known.

In [ ]:
working["full_name"] = working["full_name"].str.strip().str.title()
working["site"] = working["site"].str.strip().str.lower()
working["status"] = working["status"].str.strip().str.lower()
working[["full_name", "site", "status"]]

## Convert numeric and date types explicitly

Finite integer-valued age parses are retained; nonnumeric, infinite, and fractional values become missing without rounding. Date text must first satisfy the exact ASCII `YYYY-MM-DD` pattern and then name a possible calendar date.

In [ ]:
EXACT_DATE_PATTERN = r"[0-9]{4}-[0-9]{2}-[0-9]{2}"

age_numeric = pd.to_numeric(working["age_text"], errors="coerce")
age_finite_mask = age_numeric.notna() & age_numeric.abs().lt(float("inf"))
age_noninteger_mask = age_finite_mask & age_numeric.mod(1).ne(0)
age_integer_mask = age_finite_mask & ~age_noninteger_mask
working["age_text"] = age_numeric.where(age_integer_mask, pd.NA).astype("Int64")
working = working.rename(columns={"age_text": "age"})

exact_date_text_mask = working["visit_date"].str.fullmatch(EXACT_DATE_PATTERN, na=False)
working["visit_date"] = pd.to_datetime(
    working["visit_date"].where(exact_date_text_mask, pd.NA),
    format="%Y-%m-%d",
    errors="coerce",
)

date_contract_probe = pd.Series(["2026-01-01", "2026-1-1", "2026-02-30"], dtype="string")
date_text_mask = date_contract_probe.str.fullmatch(EXACT_DATE_PATTERN, na=False)
date_result = pd.to_datetime(
    date_contract_probe.where(date_text_mask, pd.NA),
    format="%Y-%m-%d",
    errors="coerce",
)
assert date_text_mask.tolist() == [True, False, True]
assert date_result.notna().tolist() == [True, False, False]
working.dtypes

## Apply the recorded duplicate rule and preserve uncertainty

The keep mask removes only the exact raw repeated submission. A review flag retains rows whose age or date remains uncertain.

In [ ]:
working = working.loc[exact_duplicate_keep_mask].copy()
working["needs_review"] = working["age"].isna() | working["visit_date"].isna()
review_queue = working.loc[
    working["needs_review"],
    ["record_id", "age", "visit_date"],
].copy()
review_queue

## Verify the targeted result

These checks prove that raw data remains unchanged, the exact duplicate rule removed one row, types match the contract, and uncertain values were neither rounded nor filled from adjacent people.

In [ ]:
assert raw.equals(raw_snapshot)
assert len(working) == 5
assert working["record_id"].is_unique
assert str(working["age"].dtype) == "Int64"
assert pd.api.types.is_datetime64_any_dtype(working["visit_date"].dtype)
assert working.loc[working["record_id"].eq("R002"), "age"].isna().all()
assert working.loc[working["record_id"].eq("R002"), "visit_date"].isna().all()
assert working.loc[working["record_id"].eq("R004"), "visit_date"].isna().all()
assert working.loc[working["record_id"].eq("R004"), "needs_review"].all()
assert working["site"].tolist() == ["north", "north", "south", "south", "west"]

fractional_probe = pd.to_numeric(pd.Series(["40.5"]), errors="coerce")
fractional_is_integer = fractional_probe.mod(1).eq(0)
fractional_result = fractional_probe.where(fractional_is_integer, pd.NA).astype("Int64")
assert fractional_result.isna().all()

print("Demo 2 targeted transformations verified")
working